# 📊 E-Commerce Recommendation & Ranking — Exploratory Data Analysis (EDA)

This notebook provides in-depth exploratory data analysis for the e-commerce recommendation system.
Key objectives:
1. **Behavioral Funnel Analysis**: Examine interaction distribution across `view`, `click`, `cart`, `fav`, and `buy`.
2. **Long-Tail Sparsity**: Inspect Pareto distributions for user activity and item popularity.
3. **Temporal Dynamics**: Analyze hourly and weekly interaction patterns.
4. **Feature & Label Sanity**: Check target label formulation and ensure zero data leakage across train/valid/test splits.

## 1. Environment Setup & Data Loading

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set project root
root = Path('..').resolve()
raw_path = root / 'data/raw/interactions.csv'
processed_dir = root / 'data/processed'

print(f"Project root: {root}")
print(f"Raw dataset exists: {raw_path.exists()}")

if not raw_path.exists():
    raise FileNotFoundError("Please run: python scripts/run_demo_data.py")

df = pd.read_csv(raw_path)
print(f"Total raw interactions: {len(df):,}")
df.head()

## 2. Dataset Overview & Basic Statistics

In [ ]:
stats = {
    "Total Rows": len(df),
    "Unique Users": df['user_id'].nunique(),
    "Unique Items": df['item_id'].nunique(),
    "Unique Categories": df['category_id'].nunique(),
    "Unique Brands": df['brand_id'].nunique(),
    "Interaction Density": len(df) / (df['user_id'].nunique() * df['item_id'].nunique()),
}

for k, v in stats.items():
    print(f"{k}: {v:.6f}" if isinstance(v, float) else f"{k}: {v:,}")

# Missing value check
print("\nMissing values per column:")
print(df.isnull().sum())

## 3. Behavioral Funnel & Conversion Rates

In e-commerce systems, user actions follow an extreme funnel: views dominate, while purchases are scarce.

In [ ]:
behavior_counts = df['behavior_type'].value_counts()
behavior_pct = df['behavior_type'].value_counts(normalize=True) * 100

funnel_df = pd.DataFrame({
    'Count': behavior_counts,
    'Percentage (%)': behavior_pct
})
display(funnel_df)

# Positive actions (CTR ranking label)
positive_behaviors = {'click', 'cart', 'fav', 'buy'}
pos_count = df['behavior_type'].isin(positive_behaviors).sum()
print(f"Positive interactions (Label=1): {pos_count:,} ({pos_count / len(df) * 100:.2f}%)")
print(f"Negative interactions (Label=0, Views): {len(df) - pos_count:,} ({(len(df) - pos_count) / len(df) * 100:.2f}%)")

## 4. User Activity & Item Popularity (Long-Tail Analysis)

Analyzing the Pareto distribution (80/20 rule) in user engagement and item exposure.

In [ ]:
user_activity = df.groupby('user_id').size()
item_popularity = df.groupby('item_id').size()

print("--- User Activity Statistics ---")
print(user_activity.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]))

print("\n--- Item Popularity Statistics ---")
print(item_popularity.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]))

# Top 10% items share of total interactions
top_10pct_cutoff = int(len(item_popularity) * 0.1)
top_items_share = item_popularity.sort_values(ascending=False).head(top_10pct_cutoff).sum() / len(df) * 100
print(f"\nTop 10% most popular items account for {top_items_share:.2f}% of all interactions.")

## 5. Temporal Patterns (Hour-of-Day & Day-of-Week)

Evaluating peak shopping hours and weekend effects for feature engineering.

In [ ]:
df['dt'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['dt'].dt.hour
df['day_of_week'] = df['dt'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

hourly_counts = df.groupby('hour')['behavior_type'].count()
hourly_buy_rate = df.groupby('hour')['behavior_type'].apply(lambda x: (x == 'buy').mean() * 100)

temporal_df = pd.DataFrame({
    'Total Interactions': hourly_counts,
    'Purchase Rate (%)': hourly_buy_rate
})
display(temporal_df.head(12))

## 6. Processed Feature Split Verification (Leakage Prevention Check)

Verify that the time-based split preserves temporal ordering ($Train < Valid < Test$) and that valid/test features don't leak future data.

In [ ]:
train_file = processed_dir / 'train.csv'
valid_file = processed_dir / 'valid.csv'
test_file = processed_dir / 'test.csv'

if train_file.exists() and valid_file.exists() and test_file.exists():
    train = pd.read_csv(train_file, usecols=['timestamp', 'label'])
    valid = pd.read_csv(valid_file, usecols=['timestamp', 'label'])
    test = pd.read_csv(test_file, usecols=['timestamp', 'label'])

    print(f"Train set: {len(train):,} rows | Label=1: {train['label'].mean():.4f} | Time range: {train['timestamp'].min()} ~ {train['timestamp'].max()}")
    print(f"Valid set: {len(valid):,} rows | Label=1: {valid['label'].mean():.4f} | Time range: {valid['timestamp'].min()} ~ {valid['timestamp'].max()}")
    print(f"Test set:  {len(test):,} rows | Label=1: {test['label'].mean():.4f} | Time range: {test['timestamp'].min()} ~ {test['timestamp'].max()}")
    
    # Time order assertion
    assert train['timestamp'].max() <= valid['timestamp'].min(), "Leakage detected: Train timestamps overlap Valid!"
    assert valid['timestamp'].max() <= test['timestamp'].min(), "Leakage detected: Valid timestamps overlap Test!"
    print("\n[PASSED] Strict temporal ordering verified: Train < Valid < Test (Zero temporal leakage).")
else:
    print("Processed feature splits not found. Run: python scripts/run_data_pipeline.py")